# 03 — Australia Murray-Darling Basin, HLS with Prithvi Engine

Delineates large-scale irrigated agricultural fields in the Murray-Darling Basin using **Harmonized Landsat-Sentinel (HLS)** imagery and the **Prithvi** foundation model in embedding mode.

**Estimated runtime:** ~45–90 minutes (3 years, GPU recommended).

**Prerequisites:**
```bash
pip install agribound[gee,prithvi]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/australia_murray_darling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2022, 2025)
SOURCE = "hls"
ENGINE = "prithvi"

## Create Study Area

An AOI in the Murray-Darling Basin covering large irrigated agriculture.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [145.50, -35.00],
                        [145.65, -35.00],
                        [145.65, -34.85],
                        [145.50, -34.85],
                        [145.50, -35.00],
                    ]
                ],
            },
            "properties": {"name": "Murray-Darling Basin AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "murray_darling_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation (2022–2024)

In [ ]:
all_results = {}

for year in YEARS:
    print(f"Processing {year}...", end=" ")
    output_path = OUTPUT_DIR / f"fields_hls_{year}.gpkg"

    gdf = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=year,
        engine=ENGINE,
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        composite_method="median",
        min_area=5000,
        simplify=3.0,
    )
    all_results[year] = gdf
    print(f"{len(gdf)} fields")

## Visualization

In [ ]:
if all_results:
    from agribound.visualize import show_comparison

    m = show_comparison(
        list(all_results.values()),
        labels=[str(y) for y in all_results.keys()],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_murray_darling.html"),
    )
    m